# CWL Execution - *Workflow* Class 

link to documentation: https://eoap.github.io/quickwin/cwl-workflow/cloud-native/

## CWL Inspection
Add the `Workflow` Class on top of the `CommandLineTool` Class, in order to build up the full Application workflow

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e . -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


There's a new element `$graph`, it's there to list all classes of the CWL document. In our case, one `Workflow` and one `CommandLineTool`:

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[].class' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Let's inspect the `Workflow` element keys:

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0] | keys' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


The Workflow must have an `id`, a `label` and a `doc`:

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0] | {"id": .id, "label": .label, "doc": .doc }' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


The `inputs` mapping defines all the workflow inputs. 

These are then mapped to the CommandLineTool in the `step` mapping

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0].inputs' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


The `steps` mapping defines a single step `node_detect` that runs the CWL element with id `detect-water-body`

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0].steps' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0].steps.node_detect.run' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


The mapping between the Workflow class inputs and the CommandLineTool inputs is done in the `in` dictionary: 

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0].steps.node_detect.in' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Finally the outputs are defined with:

- an `id` 
- an `outputSource` that is linked to the CommandLineTool output
- a `type`, the same as the CommandLineTool output type

In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[0].outputs' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[1].outputs' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


In [ ]:
(
set -euo pipefail
cat ../cwl-workflow/app-water-body-cloud-native.cwl | yq e .'$graph[1].outputs.stac-catalog.type' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


## CWL Execution

Let's use a CWL runner to run this CWL document.

First create the `params.yaml` file:


In [ ]:
(
set -euo pipefail
cat << EOF > params.yaml
item:
  value: https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A
aoi:
  west: -121.399
  south: 39.834
  east: -120.74
  north: 40.472
epsg: "4326"
bands:
- green
- nir
EOF
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Then use the `cwltool` to run the CWL document with the parameters:

In [ ]:
(
set -euo pipefail
cwltool --podman --outdir "${QUICKWIN_WORKSPACE:-/workspace/quickwin}/runs/cwl-workflow" ../cwl-workflow/app-water-body-cloud-native.cwl params.yaml > cwl-workflow-output.json
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Inspect the `cwl-workflow-output.json` file keys:

In [ ]:
(
set -euo pipefail
cat cwl-workflow-output.json | yq e '. | keys' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Now get the root folder of the results:

In [ ]:
(
set -euo pipefail
cat cwl-workflow-output.json | yq e '.stac_catalog.location' -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


We know there's a catalog.json file there:


In [ ]:
(
set -euo pipefail
stac describe $( cat cwl-workflow-output.json | yq e '.stac_catalog.location' - | sed 's|file://||g' )/catalog.json
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi
